# Import libraries

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
import importlib
import MLUtils
importlib.reload(MLUtils)
from MLUtils import loadModel, predict
from itertools import product
import matplotlib.pyplot as plt

# Data frame configurations

In [ ]:
pd.set_option("display.width", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", False)
pd.options.display.float_format = '{:.6g}'.format

# Project directory

In [ ]:
currentDir = Path.cwd().resolve()

projectDir = next(
    (path for path in [currentDir, *currentDir.parents] if (path / '.git').exists()),
    None
)

# Loading trained model

In [ ]:
# loading models and model information
modelsPath = projectDir/'ML'/'models'
CdModel, CdLikelihood, trainedModelDict = loadModel('Cd', modelsPath)
ClModel, ClLikelihood, _ = loadModel('Cl', modelsPath)

XMin, XMax = trainedModelDict['XMin'], trainedModelDict['XMax']

# Cross-Validation results

In [ ]:
# path to saved raw data
MLValResultsPath = projectDir/'ML'/'data'/'MLValResults.txt'
MLValResults = pd.read_csv(MLValResultsPath, sep = '\t')

# set up dictionary to save processed data
avgMLValResults = {
    'depVar' : [], 
    'kernel' : [], 
    'ARD' : [],
    'avgTestSRMSE' : [],
    'avgTestCRMSE' : [], 
    'avgTestSMAE' : [], 
    'avgTestCMAE' : [], 
    'avgTestSNLPD' : [], 
    'avgTestCNLPD' : []
}

for depVar in ['Cd', 'Cl']:
    for ARD in [None, 5]:
        for kernel in ['rbf', 'matern32', 'matern52']:
            avgMLValResults['depVar'].append(depVar)
            avgMLValResults['kernel'].append(kernel)
            avgMLValResults['ARD'].append(ARD)

# evaluate averaged scores across folds
for i in range (0, len(MLValResults['depVar'])-4, 5):
    avgMLValResults['avgTestSRMSE'].append(np.average(MLValResults['testSRMSE'][i:i+5]))
    avgMLValResults['avgTestCRMSE'].append(np.average(MLValResults['testCRMSE'][i:i+5]))
    avgMLValResults['avgTestSMAE'].append(np.average(MLValResults['testSMAE'][i:i+5]))
    avgMLValResults['avgTestCMAE'].append(np.average(MLValResults['testCMAE'][i:i+5]))
    avgMLValResults['avgTestSNLPD'].append(np.average(MLValResults['testSNLPD'][i:i+5]))
    avgMLValResults['avgTestCNLPD'].append(np.average(MLValResults['testCNLPD'][i:i+5]))

# convert to data frame for presentation
avgMLValResultsDf = pd.DataFrame(avgMLValResults)

print (avgMLValResultsDf.to_string(index = False, col_space = 15))

# scoring to find best kernel and ARD
CdScoring = {}

for kernel in ['rbf', 'matern32', 'matern52']:
    for ARD in [None, 5]:
        CdScoring[f'{kernel}/{ARD}'] = 0

CdResultsDf = (avgMLValResultsDf[avgMLValResultsDf['depVar'] == 'Cd']).copy().reset_index(drop = True)  # extract Cd results

for key in list(avgMLValResults.keys())[3:]:  # scoring based on ranking
    CdResultsDf = CdResultsDf.sort_values(by = key, ascending = True).reset_index(drop=True)
    for i in range (6):
        kernel = CdResultsDf.loc[i, "kernel"]
        ARD = CdResultsDf.loc[i, "ARD"]

        ARDLabel = "None" if pd.isna(ARD) else f"{ARD:.0f}"
        key = f"{kernel}/{ARDLabel}"

        CdScoring[key] += i

print ()
print ('Best model(s) for Cd:')
lowestValue = min(CdScoring.values())
lowestKeys = [key for key, value in CdScoring.items() if value == lowestValue]
for i in range (len(lowestKeys)):
    print (lowestKeys[i])

ClScoring = {}

for kernel in ['rbf', 'matern32', 'matern52']:
    for ARD in [None, 5]:
        ClScoring[f'{kernel}/{ARD}'] = 0

ClResultsDf = (avgMLValResultsDf[avgMLValResultsDf['depVar'] == 'Cl']).copy().reset_index(drop = True)  # extract Cl results

for key in list(avgMLValResults.keys())[3:]:  # scoring based on ranking
    ClResultsDf = ClResultsDf.sort_values(by = key, ascending = True).reset_index(drop=True)
    for i in range (6):
        kernel = ClResultsDf.loc[i, "kernel"]
        ARD = ClResultsDf.loc[i, "ARD"]

        ARDLabel = "None" if pd.isna(ARD) else f"{ARD:.0f}"
        key = f"{kernel}/{ARDLabel}"

        ClScoring[key] += i

print ()
print ('Best model(s) for Cl:')

lowestValue = min(ClScoring.values())
lowestKeys = [key for key, value in ClScoring.items() if value == lowestValue]
for i in range (len(lowestKeys)):
    print (lowestKeys[i])


# Held-out test results

In [ ]:
MLTestResultsPath = projectDir/'ML'/'data'/'MLTestResults.txt'
MLTestResults = pd.read_csv(MLTestResultsPath, sep = '\t')

print (MLTestResults.to_string(index = False, col_space = 10))

In [ ]:
# retrieve test data
testDataPath = projectDir/'ML'/'data'/'testData.txt'
testData = pd.read_csv(testDataPath, sep = '\t')

# CFD Cd and Cl data
yCdTest = testData['Cd'].to_numpy()
yClTest = testData['Cl'].to_numpy()

# predict ML Cd and Cl data
XTest = testData[['M', 'P', 'T', 'AoA']].to_numpy()

CdMean, _, ClMean, _, _, _ = predict(  # predict using models
    XTest[:, 0], XTest[:, 1], XTest[:, 2], XTest[:, 3],
    CdModel, CdLikelihood, ClModel, ClLikelihood, XMin, XMax
)

CdMean = CdMean.detach().numpy()
ClMean = ClMean.detach().numpy()

# plot 
fig, axes = plt.subplots(2, 1, figsize=(8, 10))

axes[0].scatter(yCdTest, CdMean)

CdLimits = [yCdTest.min(), yCdTest.max()]

axes[0].plot(CdLimits, CdLimits, '--', label='Perfect prediction')
axes[0].set_xlabel('CFD $C_D$')
axes[0].set_ylabel('Predicted $C_D$')
axes[0].set_title('(a) $C_D$ Prediction')
axes[0].legend(fontsize=7)
axes[0].grid(alpha=0.3)

axes[1].scatter(yClTest, ClMean)

ClLimits = [yClTest.min(), yClTest.max()]

axes[1].plot(ClLimits, ClLimits, '--', label='Perfect prediction')
axes[1].set_xlabel('CFD $C_L$')
axes[1].set_ylabel('Predicted $C_L$')
axes[1].set_title('(b) $C_L$ Prediction')
axes[1].legend(fontsize=7)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Monotonicity test

In [ ]:
# set up corner test points
monotonicityTestPoints = np.vstack((
    np.column_stack((np.zeros((2, 1)), np.full(2, 0), np.array(list(product([XMin[2], XMax[2]]))))),
    np.column_stack((np.full(4, 6), np.array(list(product([XMin[1], XMax[1]], [XMin[2], XMax[2]])))))
))

labels = [['symmetric, thin'], ['symmetric, thick'], ['forward-cambered, thin'], ['forward-cambered, thick'], ['aft-cambered, thin'], ['aft-cambered, thick']]

monotonicityTestPoints = np.column_stack((monotonicityTestPoints, labels))

# plot Cl and Cd across AoA range at each test point
fig, axes = plt.subplots(2, 1, figsize=(8, 10))

for [M, P, T, label] in monotonicityTestPoints:
    n = 200
    AoASweep = np.linspace(0, 8, n)

    CdMean, _, ClMean, _, _, _ = predict(  # predict using models
        np.full(n, M, dtype = float), np.full(n, P, dtype = float), np.full(n, T, dtype = float), AoASweep,
        CdModel, CdLikelihood, ClModel, ClLikelihood, XMin, XMax
    )
    
    CdMean = CdMean.detach().numpy()
    ClMean = ClMean.detach().numpy()
    
    axes[0].plot(AoASweep, CdMean, label=label)
    axes[1].plot(AoASweep, ClMean, label=label)

axes[0].set_xlabel('$AoA$ (degrees)')
axes[0].set_ylabel('Predicted $C_D$')
axes[0].set_title('(a) $C_D$ vs $AoA$')
axes[0].legend(fontsize=7)
axes[0].grid(alpha=0.3)

axes[1].set_xlabel('$AoA$ (degrees)')
axes[1].set_ylabel('Predicted $C_L$')
axes[1].set_title('(b) $C_L$ vs $AoA$')
axes[1].legend(fontsize=7)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()